# **ÚESz PDF to TXT to JSON conversion pipeline**

In [ ]:
""" IMPORTS """

import os
import sys
import subprocess
import shutil
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "uesz_extract":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# hu.hfstol is expected inside the repo by default
os.environ.setdefault(
    "HU_HFSTOL",
    str(PROJECT_ROOT / "input_files" / "hu.hfstol"),
)


HU_HFSTOL = PROJECT_ROOT / "input_files" / "hu.hfstol"

HFST_LOOKUP_PATH = os.environ.get("HFST_LOOKUP")

if HFST_LOOKUP_PATH is None:
    hfst_from_path = shutil.which("hfst-lookup.exe") or shutil.which("hfst-lookup")
    if hfst_from_path:
        HFST_LOOKUP_PATH = hfst_from_path
    else:
        candidate = Path.home() / "hfst" / "bin" / "hfst-lookup.exe"
        if candidate.exists():
            HFST_LOOKUP_PATH = str(candidate)

if HFST_LOOKUP_PATH is None:
    raise FileNotFoundError(
        "hfst-lookup was not found. Set HFST_LOOKUP before running this notebook, "
        "or add the HFST bin folder to PATH."
    )

from helpers import etymology_analysis_tables as eat
from helpers import pdf_to_txt
from helpers import append_wiktionary_english_loans as wik
from helpers import export_etymology_csv
from helpers import emmorph_pos
from helpers import ipa_enrichment
from helpers import semantic_features
from helpers import process_uesz_txt_files as pu
from helpers import ueszweb_abbreviations as ab
from helpers import etymology_parser as ep

In [ ]:
# THE GITHUB REPO ROOT FOLDER
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "uesz_extract":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))  # for importing helpers

# THIS NOTEBOOK'S WORKING FOLDER
EXTRACTION_DIR = PROJECT_ROOT / "uesz_extract"

# THE ORIGINAL PDF FILES ABOUT THE ETYMOLOGICAL DICTIONARY
PDF_DIR = EXTRACTION_DIR / "uesz_pdf"

# ETYMOLOGICAL DICTIONARY OUTPUT FOLDER FOR THE JSON FILES AND THE SUMMARIZED CSV FILE
OUTPUT_DIR = EXTRACTION_DIR / "uesz_output"

# PDF-BASED EXTRACTION OUTPUTS
TXT_OUTPUT_DIR = OUTPUT_DIR / "pdf_marked_txt"
JSON_OUTPUT_DIR = OUTPUT_DIR / "pdf_structured_json"

# CSV FILE OUTPUT
WIKTIONARY_CSV = PROJECT_ROOT / "input_files" / "wiktionary_hu_borrowed_from_english_manually_grouped.csv"
CSV_UESZ_DB = OUTPUT_DIR / "uesz_structured_db.csv"

# Create output folders
PDF_DIR.mkdir(parents=True, exist_ok=True)
JSON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TXT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("PDF folder:", PDF_DIR)
print("Output folder:", OUTPUT_DIR)
print("Structured CSV:", CSV_UESZ_DB)


In [ ]:
RUN_PDF_TO_TXT = True

if RUN_PDF_TO_TXT:
    for pdf_file in PDF_DIR.glob("*.pdf"):
        pdf_to_txt.process_one_pdf(pdf_file, TXT_OUTPUT_DIR)

# **TXT TO JSON EXTRACTION**

In [ ]:
"""
EXTRACTION OF DICTIONARY ARTICLES INTO JSON FILES
"""
 # change to True if it needs a re-run
RUN_TXT_TO_JSON = True

if RUN_TXT_TO_JSON:
    pu.process_all_uesz_txt_files(input_dir=TXT_OUTPUT_DIR, output_dir=JSON_OUTPUT_DIR)

# **JSON FILES TO CSV DATABASE EXTRACTION**


In [ ]:
""" CSV pipeline in seven steps, with wiktionary English entries
and semantic enrichment (in addition to POS and IPA)"""

""" CHANGE TO 'False' IF RELEVANT 
ENRICHMENT PROCESS IN CSV DOES NOT NEED TO RE-RUN """
ADD_EMMORPH_POS = True
ADD_IPA = True
ADD_SEMANTIC_FIELDS = True
DELETE_TEMP_FILES = True
ADD_WIKTIONARY_ENGLISH = True

wiktionary_csv = CSV_UESZ_DB.with_name(f"{CSV_UESZ_DB.stem}_wiktionary_tmp.csv")
base_csv = CSV_UESZ_DB.with_name(f"{CSV_UESZ_DB.stem}_base_tmp.csv")
pos_csv = CSV_UESZ_DB.with_name(f"{CSV_UESZ_DB.stem}_pos_tmp.csv")
ipa_csv = CSV_UESZ_DB.with_name(f"{CSV_UESZ_DB.stem}_ipa_tmp.csv")
sem_csv = CSV_UESZ_DB.with_name(f"{CSV_UESZ_DB.stem}_semantic_tmp.csv")

# 1. export csv
cmd = [
    sys.executable,
    "-B",
    "export_etymology_csv.py",
    "--json-root",
    str(OUTPUT_DIR),
    "--output",
    str(base_csv),
]
subprocess.run(cmd, cwd=Path.cwd().parent / "helpers", text=True, capture_output=True, check=True)

current_csv = base_csv

# 2. add POS
if ADD_EMMORPH_POS:
    ger_tagger = emmorph_pos.make_spacy_german_pos_tagger("de_core_news_sm")

    emmorph_pos.add_pos_to_csv(
        input_csv=current_csv,
        output_csv=pos_csv,
        hfst_lookup=HFST_LOOKUP_PATH,
        hu_hfstol=PROJECT_ROOT / "input_files" / "hu.hfstol",
        german_pos_tagger=ger_tagger,
        keep_debug_columns=False,
    )

    current_csv = pos_csv

    
# 3. add manually selected Wiktionary English loanwords
if ADD_WIKTIONARY_ENGLISH:

    wik.append_wiktionary_english_loans(
        input_csv=current_csv,
        output_csv=wiktionary_csv,
        grouped_csv=WIKTIONARY_CSV,
        start_line=301,
        end_line=376,
        expected_count=76,
    )
    current_csv = wiktionary_csv


# 4. add IPA
if ADD_IPA:

    ipa_enrichment.add_ipa_to_csv(
        input_csv=current_csv,
        output_csv=ipa_csv,
    )
    current_csv = ipa_csv

# 5. add semantic fields
if ADD_SEMANTIC_FIELDS:

    df = pd.read_csv(current_csv, encoding="utf-8-sig", dtype=str).fillna("")
    df = ipa_enrichment.normalize_integerish_columns(df)
    df = semantic_features.assign_semantic_fields(df, output_col="semantic_field_odenet")
    df = ipa_enrichment.normalize_integerish_columns(df)
    df.to_csv(sem_csv, index=False, encoding="utf-8-sig")
    current_csv = sem_csv

# 6. save final file
if current_csv != CSV_UESZ_DB:
    df = pd.read_csv(current_csv, encoding="utf-8-sig", dtype=str).fillna("")
    df = ipa_enrichment.normalize_integerish_columns(df)
    df.to_csv(CSV_UESZ_DB, index=False, encoding="utf-8-sig")

print(f"Final CSV ready: {CSV_UESZ_DB}")

# 7. delete temp files
if DELETE_TEMP_FILES:
    for tmp_path in [base_csv, pos_csv, wiktionary_csv, ipa_csv, sem_csv]:
        if tmp_path.exists() and tmp_path != CSV_UESZ_DB:
            tmp_path.unlink()



### **Compare current CSV with gold thesis CSV**

In [ ]:


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"classification", "uesz_extract", "emmorph_audit"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

gold_path = PROJECT_ROOT / "input_files" / "etymology_database_gold_thesis.csv"
current_path = PROJECT_ROOT / "uesz_extract" / "uesz_output" / "uesz_structured_db.csv"

print("Gold exists:", gold_path.exists(), gold_path)
print("Current exists:", current_path.exists(), current_path)

gold = pd.read_csv(gold_path, dtype=str).fillna("")
current = pd.read_csv(current_path, dtype=str).fillna("")

base_key_cols = ["lemma", "homograph", "entry_file"]

gold = gold.copy()
current = current.copy()

gold["_row_in_entry"] = gold.groupby(base_key_cols, dropna=False).cumcount()
current["_row_in_entry"] = current.groupby(base_key_cols, dropna=False).cumcount()

key_cols = base_key_cols + ["_row_in_entry"]

shared_cols = [
    col for col in gold.columns
    if col in current.columns and col not in key_cols
]

merged = gold.merge(
    current,
    on=key_cols,
    how="inner",
    suffixes=("_gold", "_current"),
    validate="one_to_one",
)

rows = []
for col in shared_cols:
    gold_col = f"{col}_gold"
    current_col = f"{col}_current"

    diff = merged[gold_col].astype(str) != merged[current_col].astype(str)
    rows.append({
        "column_name": col,
        "n_diff": int(diff.sum()),
        "n_rows": len(merged),
        "diff_%": round(diff.mean() * 100, 2),
    })

diff_summary = (
    pd.DataFrame(rows)
    .sort_values(["n_diff", "column_name"], ascending=[False, True])
    .reset_index(drop=True)
)

display(diff_summary)

## **STATISTICS ABOUT THE GOLD CSV FILE**

In [ ]:
df_gold = pd.read_csv(gold_path)

# POS cumulative coverage tables
pos_counts, pos_pcts = emmorph_pos.emmorph_pos_cumulative_stage_table(df_gold)
display(pos_counts)
display(pos_pcts)


In [ ]:
pos_detail_counts, pos_detail_pcts = emmorph_pos.emmorph_pos_source_detail_table(df_gold)
display(pos_detail_counts)
display(pos_detail_pcts)

In [ ]:
origin_formation = eat.origin_formation_crosstab(df_gold)

display(origin_formation)

In [ ]:
semantic_counts, semantic_pcts = eat.semantic_label_by_donor_group_tables(df_gold)

display(semantic_counts)
display(semantic_pcts)